# MCP, end to end: when you need it and when you don't

We will feel the problem first, then fix it.

| Part | What happens |
|---|---|
| 1 | Write four Tavily tools by hand. It works. |
| 2 | A second project needs them too. We copy the file, and the copies drift apart. |
| 3 | MCP: those tools become one URL. |
| 4 | A simple real use case: one agent, two MCP servers, including our own. |
| 5 | When MCP is overkill. |

Everything is traced with **[Rius](https://eu.console.rius-glassflow.com/w/ac50c1f2-e7d4-522b-ba6a-0850d3e7a7d0)**.

## Setup

Add these to the `.env` file at the repo root. A free Tavily key comes from [app.tavily.com](https://app.tavily.com).

```
OPENAI_API_KEY=...
TAVILY_API_KEY=tvly-...
RIUS_API_KEY=ri_...
```

In [1]:
import os
import warnings
import fastmcp
import langchain_core
import rius
from rius import SpanKind
from dotenv import load_dotenv

load_dotenv()  # reads the .env file at the repo root
os.environ["LANGSMITH_TRACING"] = "false"  # this notebook is traced by Rius only
warnings.filterwarnings("ignore")  # hide library warnings so the output stays easy to read (set after the imports above)

for key_name in ["OPENAI_API_KEY", "TAVILY_API_KEY", "RIUS_API_KEY"]:
    print(f"{key_name:16} found: {bool(os.getenv(key_name))}")

rius_client = rius.init(service_name="langgraph-life-mcp", agent_name="langgraph-life-mcp")

MODEL_NAME = "openai:gpt-4.1-mini"

OPENAI_API_KEY   found: True
TAVILY_API_KEY   found: True
RIUS_API_KEY     found: True


## Part 1: write the tools by hand

Tavily is a web search API for AI apps. It can do four things: search the web, read a page, crawl a site and map a site.

To give those to an agent, we wrap each one in a `@tool` function. This is the normal way, and it works well.

In [2]:
from langchain.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


@tool
def search_web(query: str, max_results: int = 3) -> str:
    """Search the web for up-to-date information."""
    response = tavily_client.search(query, max_results=max_results)
    return "\n\n".join(f"{page['title']}\n{page['url']}\n{page['content']}" for page in response["results"])


@tool
def extract_page(url: str) -> str:
    """Read the text of one web page."""
    response = tavily_client.extract(url)
    return response["results"][0]["raw_content"][:2000]


@tool
def crawl_site(url: str, instructions: str) -> str:
    """Crawl a website and return the pages that match the instructions."""
    response = tavily_client.crawl(url, instructions=instructions, limit=3)
    return "\n\n".join(f"{page['url']}\n{page['raw_content'][:500]}" for page in response["results"])


@tool
def map_site(url: str) -> str:
    """List the page URLs of a website."""
    response = tavily_client.map(url, limit=10)
    return "\n".join(response["results"])


handwritten_tools = [search_web, extract_page, crawl_site, map_site]

Try one tool on its own, before giving it to an agent:

In [3]:
print(search_web.invoke({"query": "Model Context Protocol introduction for developers", "max_results": 2}))

Introducing the Model Context Protocol \ Anthropic
https://www.anthropic.com/news/model-context-protocol
The Model Context Protocol is an open standard that enables developers to build secure, two-way connections between their data sources and AI-powered tools. The architecture is straightforward: developers can either expose their data through MCP servers or build AI applications (MCP clients) that connect to these servers.

Today, we're introducing three major components of the Model Context Protocol for developers: [...] # Introducing the Model Context Protocol

 Published
 Category

  Product
 Topics

  + Model Context Protocol
  + Open source
  + Tool use
  + Claude apps
  + Partnerships

Today, we're open-sourcing the Model Context Protocol (MCP), a new standard for connecting AI assistants to the systems where data lives, including content repositories, business tools, and development environments. Its aim is to help frontier models produce better, more relevant responses. [...]

Now give all four to an agent. `@rius.observe` keeps the whole run in **one trace**.

In [4]:
from langchain.agents import create_agent


def tools_used(messages):
    """Small helper: list the tool names the agent called."""
    return [call["name"] for message in messages if message.type == "ai" for call in message.tool_calls]


research_bot = create_agent(
    name="research_bot",
    model=MODEL_NAME,
    tools=handwritten_tools,
    system_prompt="You are a research assistant. Use your tools, then answer in 2 sentences with one source URL.",
)


@rius.observe(name="Research bot (hand-written tools)", kind=SpanKind.AGENT)
def ask_research_bot(question):
    return research_bot.invoke({"messages": [{"role": "user", "content": question}]})


result = ask_research_bot("What is the Model Context Protocol?")
print("Tools :", tools_used(result["messages"]))
print("Answer:", result["messages"][-1].content)

Tools : ['search_web']
Answer: The Model Context Protocol (MCP) is an open standard that allows AI models to seamlessly communicate and integrate with external tools, data sources, and APIs via a universal protocol, simplifying integration and enabling scalable AI deployments. It acts like a common connector so any compliant AI client can access various system capabilities without custom integrations for each pair. (source: https://aibuzz.blog/model-context-protocol-explained)


## Part 2: the pain

A second team builds a **support bot**, and it needs the same Tavily tools. The quickest way is to copy the file.

```
project_research_bot/tavily_tools.py
project_support_bot/tavily_tools.py    <- copied
```

Some weeks later the research team improves their copy. Nobody remembers the other one. Let's compare the two files:

In [5]:
import difflib
from pathlib import Path

research_copy = Path("project_research_bot/tavily_tools.py").read_text().splitlines()
support_copy = Path("project_support_bot/tavily_tools.py").read_text().splitlines()

print("Lines in each copy:", len(research_copy), "and", len(support_copy), "\n")

for line in difflib.unified_diff(research_copy, support_copy, "research_bot", "support_bot", lineterm="", n=0):
    print(line)

Lines in each copy: 36 and 36 

--- research_bot
+++ support_bot
@@ -1 +1 @@
-"""Tavily tools for the research bot."""
+"""Tavily tools for the support bot. Copied from the research bot a while ago."""
@@ -12 +12 @@
-def search_web(query: str, max_results: int = 3) -> str:
+def search_web(query: str) -> str:
@@ -14 +14 @@
-    response = tavily_client.search(query, max_results=max_results)
+    response = tavily_client.search(query, max_results=5)
@@ -22 +22 @@
-    return response["results"][0]["raw_content"][:2000]
+    return response["results"][0]["raw_content"]


Same tools, different behaviour. This is what breaking **DRY** (Don't Repeat Yourself) looks like in real life. Watch both copies answer the same request:

In [6]:
from project_research_bot.tavily_tools import search_web as research_search
from project_support_bot.tavily_tools import search_web as support_search

question = {"query": "What is the Model Context Protocol?"}

print("Research bot pages:", research_search.invoke(question).count("http"))
print("Support bot pages :", support_search.invoke(question).count("http"))

Research bot pages: 3


Support bot pages : 5


### "Can't I just make a shared Python package?"

Yes, and for Python projects that is often the right answer:

```python
from company_tools.tavily import search_web   # one copy, used by both bots
```

But a package only reaches **Python code**. It stops working when:

| Who needs the tools | Shared package | MCP |
|---|---|---|
| Another Python project | Yes | Yes |
| A JavaScript app | No | Yes |
| Claude Code, Cursor and other AI apps | No | Yes |
| Tavily improves its tools | You update your code | They update their server |

That is the problem MCP solves.

## Part 3: MCP

**MCP (Model Context Protocol)** is a standard way for an AI app to talk to tools.

Think of a phone charger. Before USB-C, every phone needed its own cable. With one standard plug, any charger works with any phone. MCP is that plug for AI tools.

```
Without MCP:  every app writes its own code for every tool
With MCP:     tool makers run one server, and every app connects to it
```

| Word | Means | Here |
|---|---|---|
| **Server** | Owns the tools | Tavily's server |
| **Client** | Connects and uses them | Our agent, through `MCPAdapter` |

Tavily already runs a server, so we only need its URL. MCP calls use `await` because they travel over the network, and notebooks allow `await` directly.

Docs: [LangChain MCP](https://docs.langchain.com/oss/python/langchain/mcp) · [Tavily MCP](https://docs.tavily.com/documentation/mcp)

In [7]:
from langchain.mcp import MCPAdapter

tavily_server = {
    "url": "https://mcp.tavily.com/mcp/",
    "headers": {"Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}"},
}

async with MCPAdapter({"mcpServers": {"tavily": tavily_server}}) as tavily_adapter:
    tavily_tools = await tavily_adapter.list_tools()

for mcp_tool in tavily_tools:
    print(mcp_tool.name)

tavily_search
tavily_extract
tavily_crawl
tavily_map
tavily_research


We wrote **no tool code** and still got every tool, with its description and arguments, straight from the server. Compare the search tool's arguments:

In [8]:
mcp_search = next(mcp_tool for mcp_tool in tavily_tools if mcp_tool.name == "tavily_search")

print("Our search tool :", list(search_web.args))
print("Tavily's search :", list(mcp_search.args))

Our search tool : ['query', 'max_results']
Tavily's search : ['query', 'max_results', 'search_depth', 'topic', 'time_range', 'include_images', 'include_image_descriptions', 'include_raw_content', 'include_domains', 'exclude_domains', 'country', 'include_favicon', 'start_date', 'end_date', 'exact_match', 'include_published_date', 'filter_by_published_date']


One thing to watch: the server decides how much text a tool returns, and we can't trim it the way we did in our own tools. Reading a whole page can return a lot of text, and every word costs tokens.

So give each agent **only the tools it needs**. Here that is just search:

In [9]:
search_only_tools = [mcp_tool for mcp_tool in tavily_tools if mcp_tool.name == "tavily_search"]

print("Tools for our bots:", [mcp_tool.name for mcp_tool in search_only_tools])

Tools for our bots: ['tavily_search']


Now **both projects** connect to the same server. No copied files, so nothing can drift apart. `@rius.observe` works on `async` functions too.

In [10]:
research_bot_mcp = create_agent(
    name="research_bot_mcp",
    model=MODEL_NAME,
    tools=search_only_tools,
    system_prompt="You are a research assistant. Use your tools, then answer in 2 sentences with one source URL.",
)

support_bot_mcp = create_agent(
    name="support_bot_mcp",
    model=MODEL_NAME,
    tools=search_only_tools,
    system_prompt="You are a technical support assistant. Always search the web before you answer, then answer in 2 short sentences.",
)


@rius.observe(name="Two bots, one MCP server", kind=SpanKind.AGENT)
async def ask_both_bots():
    research = await research_bot_mcp.ainvoke({"messages": [{"role": "user", "content": "What is the Model Context Protocol?"}]})
    support = await support_bot_mcp.ainvoke({"messages": [{"role": "user", "content": "What is the latest version of the MCP specification?"}]})
    return research, support


research_result, support_result = await ask_both_bots()

print("Research bot tools:", tools_used(research_result["messages"]))
print("Research bot      :", research_result["messages"][-1].content, "\n")
print("Support bot tools :", tools_used(support_result["messages"]))
print("Support bot       :", support_result["messages"][-1].content)

Research bot tools: ['tavily_search']
Research bot      : The Model Context Protocol (MCP) is an open standard and open-source protocol introduced to enable seamless integration between AI applications, especially large language models, and external data sources, tools, and systems through a consistent interface. It acts like a universal connector (compared to USB-C for devices) that simplifies building and reusing integrations, allowing AI systems to access dynamic information, perform tasks, and have bidirectional communication with various external resources securely and efficiently. 

Source: https://aibuzz.blog/model-context-protocol-explained 

Support bot tools : ['tavily_search']
Support bot       : The latest version of the MCP specification is 2026-07-28, released on July 28, 2026. It introduces a stateless protocol and several enhancements over previous versions.


## Part 4: a simple real use case

**Use case:** a support agent needs the company's **private** tools (tickets and the knowledge base) and the **public** web. Several apps in the company will want those private tools, so we put them on our own MCP server.

```
                  +--> Tavily server    (the web)
support agent ----|
                  +--> our own server   (tickets, knowledge base)
```

Open [servers/support_server.py](servers/support_server.py). It is a normal Python file: `FastMCP` turns plain functions into MCP tools.

```python
server = FastMCP("support")

@server.tool
def get_ticket(ticket_id: str) -> str:
    ...
```

A server is its own program, so we start it as a separate process:

In [11]:
import socket
import subprocess
import sys
import time


def start_server(script, port):
    """Start an MCP server as its own process and wait until it is ready."""
    process = subprocess.Popen([sys.executable, script], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for attempt in range(60):
        try:
            socket.create_connection(("127.0.0.1", port), timeout=1).close()
            return process
        except OSError:
            time.sleep(0.5)
    raise RuntimeError(f"{script} did not start")


support_server_process = start_server("servers/support_server.py", port=8765)
print("Support server is running")

Support server is running


Connect to our server the same way, then simply add the two tool lists together. The agent doesn't care where a tool came from.

In [12]:
async with MCPAdapter("http://127.0.0.1:8765/mcp") as support_adapter:
    support_tools = await support_adapter.list_tools()

print("From our server:", [mcp_tool.name for mcp_tool in support_tools])
print("From Tavily    :", [mcp_tool.name for mcp_tool in search_only_tools])

support_agent_tools = support_tools + search_only_tools

From our server: ['lookup_knowledge_base', 'get_ticket']
From Tavily    : ['tavily_search']


`rius.session` and `rius.user` tag the trace with the conversation and the customer. Our server also calls `rius.init`, so in Rius its steps sit **inside the agent's trace**: one view across both programs.

In [13]:
from langchain.chat_models import init_chat_model

# One tool at a time, so the agent reads the ticket before it searches for the error code
one_step_model = init_chat_model(MODEL_NAME, temperature=0, model_kwargs={"parallel_tool_calls": False})

support_agent = create_agent(
    name="support_agent",
    model=one_step_model,
    tools=support_agent_tools,
    system_prompt=(
        "You are a technical support assistant. First read the ticket. Only after you have read it, "
        "search the internal knowledge base using the error code from the ticket. "
        "Use the web only for public standards. Keep the answer under 4 sentences."
    ),
)


@rius.observe(name="Support agent, two MCP servers", kind=SpanKind.AGENT)
async def handle_ticket(ticket_id):
    question = f"Help me with ticket {ticket_id}. What is the problem, what is our internal fix, and what does the error mean publicly?"
    return await support_agent.ainvoke({"messages": [{"role": "user", "content": question}]})


with rius.session("ticket-T-4049"), rius.user("priya@alice-corp.example"):
    ticket_result = await handle_ticket("T-4049")

print("Tools :", tools_used(ticket_result["messages"]))
print("Answer:", ticket_result["messages"][-1].content)

Tools : ['get_ticket', 'lookup_knowledge_base']
Answer: The problem in ticket T-4049 is that the checkout API is returning error 503, meaning the service is unavailable since 9am, causing loss of sales. Internally, this error indicates the API gateway cannot reach a healthy upstream worker. The fix is to check the worker pool on the status dashboard and restart any unhealthy worker nodes using the command `nwctl workers restart --unhealthy`. If more than 30% of workers are down, a P1 incident should be opened. Publicly, error 503 means the server is temporarily unable to handle the request, usually due to maintenance or overload.


**The real payoff:** any MCP client can now use these tools. To add them to Claude Code, for example, run this while the server is up:

```bash
claude mcp add --transport http support http://127.0.0.1:8765/mcp
```

Same server, no new code.

## Part 5: when MCP is overkill

**Use case:** one Python app needs a loan EMI calculator. There is one tool, one app and one language, and nobody else needs it.

Here it is as a plain tool:

In [14]:
@tool
def calculate_emi(loan_amount: float, annual_rate_percent: float, months: int) -> float:
    """Calculate the monthly payment (EMI) of a loan."""
    monthly_rate = annual_rate_percent / 12 / 100
    growth = (1 + monthly_rate) ** months
    return round(loan_amount * monthly_rate * growth / (growth - 1), 2)


loan = {"loan_amount": 500000, "annual_rate_percent": 9, "months": 60}
print("Plain tool EMI:", calculate_emi.invoke(loan))

Plain tool EMI: 10379.18


And here is the same thing through MCP: the same function in [servers/emi_server.py](servers/emi_server.py), a process to run, a port, a connection and `await`.

In [15]:
emi_server_process = start_server("servers/emi_server.py", port=8766)

async with MCPAdapter("http://127.0.0.1:8766/mcp") as emi_adapter:
    emi_tools = await emi_adapter.list_tools()

emi_mcp_tool = emi_tools[0]
mcp_output = await emi_mcp_tool.ainvoke(loan)
print("MCP tool EMI:", mcp_output[0]["text"])

MCP tool EMI: 10379.18


Same answer. Let's measure what MCP costs here:

In [16]:
import inspect

CALLS = 20


@rius.observe(name="EMI: plain tool vs MCP", kind=SpanKind.CHAIN)
async def compare_emi_speed():
    start = time.perf_counter()
    for call in range(CALLS):
        calculate_emi.invoke(loan)
    plain_ms = (time.perf_counter() - start) / CALLS * 1000

    start = time.perf_counter()
    for call in range(CALLS):
        await emi_mcp_tool.ainvoke(loan)
    mcp_ms = (time.perf_counter() - start) / CALLS * 1000
    return plain_ms, mcp_ms


plain_ms, mcp_ms = await compare_emi_speed()
plain_lines = len(inspect.getsource(calculate_emi.func).splitlines())
server_lines = len(Path("servers/emi_server.py").read_text().splitlines())

print(f"Plain tool: {plain_ms:8.2f} ms per call | {plain_lines} lines of code | no extra process")
print(f"MCP tool  : {mcp_ms:8.2f} ms per call | {server_lines} lines in a server file, plus connection code | 1 extra process")
print(f"MCP is about {mcp_ms / plain_ms:.0f} times slower here")

Plain tool:     1.13 ms per call | 6 lines of code | no extra process
MCP tool  :    33.04 ms per call | 16 lines in a server file, plus connection code | 1 extra process
MCP is about 29 times slower here


For one small tool, MCP adds a process to keep alive and a network hop, and gives nothing back. Nobody else is sharing this tool.

## So when should you use MCP?

| Your situation | Use |
|---|---|
| One app, one language, a few small tools | Plain `@tool` |
| Several Python projects share tools | A shared Python package, or MCP |
| Different languages or AI apps need the same tools | **MCP** |
| A vendor already runs a server (like Tavily) | **MCP** |
| One team owns tools that other teams use | **MCP** |

**Simple rule:** start with `@tool`. Move to MCP when a second kind of app needs the same tools.

## Finish up

This stops both servers. We wait a few seconds first, because the support server sends its traces in the background.

In [17]:
time.sleep(10)
support_server_process.terminate()
emi_server_process.terminate()

print("All spans delivered to Rius:", rius_client.flush())

All spans delivered to Rius: True


## What to look at in Rius

Open your **[Rius console](https://eu.console.rius-glassflow.com/w/ac50c1f2-e7d4-522b-ba6a-0850d3e7a7d0)**:

| Trace | What to notice |
|---|---|
| Research bot (hand-written tools) | Our `search_web` tool inside the agent |
| Two bots, one MCP server | Both bots calling Tavily's search tool, each call shown as `execute_tool tavily_search` |
| Support agent, two MCP servers | Steps from **our server** nested inside the agent's trace · session `ticket-T-4049` · user `priya@alice-corp.example` |
| EMI: plain tool vs MCP | 20 MCP calls, each with its own network step |

The Agents view shows `support-mcp-server` as its own agent, because it is its own program.

**Try it yourself**
1. Add a `close_ticket` tool to `servers/support_server.py`, restart it, and see the agent find it with no client change.
2. Connect the support server to Claude Code with the command in Part 4.
3. Rewrite Part 5 with your own small tool. Would MCP be worth it there?